# 🏥 CariSurg MedTech Pathways — Week 0, Tutorial 1
## Python for Basic Data Exploration & Cleaning
**Mercer General Hospital | Clinical AI & Innovation Unit**

---
> **Scenario:** I have just joined the Clinical AI & Innovation Unit at Mercer General Hospital.
> Dr. De Freitas has asked you to get your tools in place and run a first look at a sample triage dataset.
> Today we focus on getting set up and tackling Day 1's challenge: cleaning the Gender column.

---
### What was covered today
1. Google Colab environment walkthrough
2. Python basics — variables, data types, lists, loops, conditionals, functions
3. Importing libraries and loading a CSV
4. Cleaning the `Gender` column (Day 1 challenge)

**Day 1 Submission:** A Jupyter notebook showing my cleaned `Gender` column — pushed to my `carisurg-portfolio` GitHub repo.


## 1. Google Colab Environment

### **Environment Setup**

This section confirms that the notebook is running successfully in Google Colab. I mounted Google Drive so the dataset can be accessed from my Drive folder, and I checked the Python version to confirm that the environment meets the programme requirement of Python 3.10 or higher.


In [1]:
# Mount your Google Drive so files are accessible from Colab
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted successfully!")

Mounted at /content/drive
Drive mounted successfully!


In [2]:
# Always confirm your Python version first
import sys
print(f"Python version: {sys.version}")
# We need 3.10 or higher for this programme

Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


## 2. Importing Libraries and Loading the Dataset

### **Load the Week 0 Dataset**

This section loads the Week 0 CSV file into the notebook using pandas. Since my dataset is stored in Google Drive, I used the Drive file path instead of uploading the CSV directly into Colab session storage.

In [3]:
# The three libraries we use most in this programme
import pandas as pd          # data manipulation
import numpy as np           # numerical operations
import matplotlib.pyplot as plt  # plotting

print("Libraries imported successfully!")

Libraries imported successfully!


This will vary depending on whether you are using Python locally or through Google Collab.

In [4]:
# ── Path adjusted to match where my files are located ──────────
#FILE_PATH = 'EmergencyTriageDataset_Reduced_Dirty.csv'
FILE_PATH = '/content/drive/MyDrive/Carisurg Portfolio/Week 0/Data/EmergencyTriageDataset_Reduced_Dirty.csv'

df_raw = pd.read_csv(FILE_PATH)
print(f"Dataset loaded: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns")
print(f"Columns: {list(df_raw.columns)}")

Dataset loaded: 2205 rows x 11 columns
Columns: ['ID', 'Age', 'Gender', 'GCS', 'SBP', 'DBP', 'MAP', 'pulse', 'Temp', 'RR', 'Fio2']


In [5]:
# Looking at the first few rows immediately after loading
df_raw.head(10)

,ID,Age,Gender,GCS,SBP,DBP,MAP,pulse,Temp,RR,Fio2
0,1,34,0,15.0,93,67.0,75.67,128.0,36.8,14.0,21.0
1,2,20,Male,15.0,130,90.0,103.33,80.0,37.0,16.0,21.0
2,3,77,Female,14.0,163,105.0,124.33,92.0,36.8,18.0,21.0
3,4,23,0,8.0,100,60.0,73.33,100.0,37.0,12.0,100.0
4,5,86,FEMALE,15.0,150,90.0,110.00,85.0,37.0,19.0,21.0
5,6,42,Male,15.0,100,60.0,73.33,99.0,37.0,20.0,21.0
6,7,75,Female,15.0,120,80.0,93.33,99.0,37.0,25.0,21.0
7,8,25,Male,15.0,100,50.0,66.67,85.0,37.0,25.0,21.0
8,9,67,0,15.0,110,70.0,83.33,78.0,37.0,16.0,21.0
9,11,82,Female,15.0,153,82.0,105.67,130.0,37.0,19.0,21.0


In [6]:
# What data types does pandas think each column is?
print(df_raw.dtypes)

ID          int64
Age         int64
Gender     object
GCS        object
SBP        object
DBP       float64
MAP       float64
pulse      object
Temp       object
RR        float64
Fio2      float64
dtype: object


### **Initial Data Quality Check**

After loading the dataset, I reviewed the column data types to identify possible data quality issues. Some columns that are expected to contain numeric clinical values may appear as `object` data types, which usually means they are being stored as strings.

This can happen in real hospital datasets when values are entered inconsistently, include extra characters, contain missing entries, or are recorded using different formats. Identifying these issues early helps guide the cleaning process.

## 3. Day 1 Challenge — Cleaning the Gender Column

### **Explore the Gender Column Before Cleaning**
Before cleaning the dataset, I first inspected the original `Gender` column to understand how the values were recorded. This step is important because data should not be modified until the existing patterns, inconsistencies, and missing values have been reviewed.


In [7]:
# Step 1: What values are actually in the Gender column?
print("Unique values in Gender:")
print(df_raw['Gender'].unique())
print(f"\nTotal unique values: {df_raw['Gender'].nunique()}")

Unique values in Gender:
['0' 'Male' 'Female' 'FEMALE' '1' 'MALE']

Total unique values: 6


### **Gender Encoding Decision**

After inspecting the original `Gender` column, I observed several variants of the same categories, including `0`, `1`, `Male`, `MALE`, `Female`, and `FEMALE`.

For this task, I used integer encoding where `1 = Male` and `0 = Female`, following the programme’s expected format. This approach is useful because numeric values are easier to process in later data analysis or modelling tasks. However, text labels such as `Male` and `Female` are more readable for humans.

To make the cleaning process more robust, I also accounted for possible edge cases. Non-binary or other gender-diverse entries would be coded as `2`, while missing, unclear, or unexpected entries would be coded as `-1` for `Unknown/Unspecified`.

This is a design decision, so I documented the coding scheme clearly to ensure the cleaned dataset remains understandable.

In [8]:
# Step 2: Count how many of each value we have
print("Gender value counts:")
print(df_raw['Gender'].value_counts())

Gender value counts:
Gender
1         422
MALE      379
Male      375
FEMALE    366
Female    340
0         323
Name: count, dtype: int64


In [9]:
# Step 3: Define a cleaning function that standardizes Gender values
# and handles binary, non-binary/other, missing, and unexpected entries.
def clean_gender(value):
    """
    This function standardizes the Gender column into consistent categories.

    Categories used:
    - 1 = Male
    - 0 = Female
    - 2 = Non-binary/Other
    - -1 = Unknown/Unspecified
    """

    # This conditional would handle missing values using the pandas function
    if pd.isna(value):
        return -1

    # Convert value to a clean lowercase string
    value = str(value).strip().lower()

    # Handle blank or missing-like entries
    if value in ["", "nan", "none", "null", "n/a", "na"]:
        return -1

    # Male entries
    if value in ["1", "m", "male", "man", "boy"]:
        return 1

    # Female entries
    if value in ["0", "f", "female", "woman", "girl"]:
        return 0

    # Non-binary / other gender-diverse entries
    if value in [
        "non-binary", "non binary", "nonbinary", "nb",
        "other", "genderqueer", "gender diverse", "agender"
    ]:
        return 2

    # Declined, unclear, or not recorded
    if value in [
        "unknown", "unspecified", "prefer not to say",
        "declined", "not stated", "not recorded"
    ]:
        return -1

    # Catch any unexpected entry
    return -1


# Step 4: Apply the cleaning function to create a new clean column
df_raw["Gender_Clean"] = df_raw["Gender"].apply(clean_gender)

# Step 5: Verify the cleaned column
print("After cleaning:")
print(df_raw["Gender_Clean"].value_counts(dropna=False))
print(f"\nAny NaN values? {df_raw['Gender_Clean'].isnull().sum()}")

After cleaning:
Gender_Clean
1    1176
0    1029
Name: count, dtype: int64

Any NaN values? 0


In [10]:
# Step 6: Drop the original dirty column now that we have a clean one
df_raw = df_raw.drop(columns=['Gender'])
df_raw = df_raw.rename(columns={'Gender_Clean': 'Gender'})

# Confirm the result
print("Column 'Gender' after cleaning:")
print(df_raw['Gender'].value_counts(dropna=False))
print(f"Data type: {df_raw['Gender'].dtype}")
df_raw.head()

Column 'Gender' after cleaning:
Gender
1    1176
0    1029
Name: count, dtype: int64
Data type: int64


,ID,Age,GCS,SBP,DBP,MAP,pulse,Temp,RR,Fio2,Gender
0,1,34,15.0,93,67.0,75.67,128.0,36.8,14.0,21.0,0
1,2,20,15.0,130,90.0,103.33,80.0,37.0,16.0,21.0,1
2,3,77,14.0,163,105.0,124.33,92.0,36.8,18.0,21.0,0
3,4,23,8.0,100,60.0,73.33,100.0,37.0,12.0,100.0,0
4,5,86,15.0,150,90.0,110.00,85.0,37.0,19.0,21.0,0


In [11]:
# Final fix: convert numeric Gender codes to human-readable labels
# This improves readability based on feedback.

gender_label_mapping = {
    1: "Male",
    0: "Female"
}

df_raw["Gender"] = df_raw["Gender"].map(gender_label_mapping)

# Confirm the result
print("Column 'Gender' after converting to readable labels:")
print(df_raw["Gender"].value_counts(dropna=False))
print(f"Data type: {df_raw['Gender'].dtype}")

df_raw.head()

Column 'Gender' after converting to readable labels:
Gender
Male      1176
Female    1029
Name: count, dtype: int64
Data type: object


,ID,Age,GCS,SBP,DBP,MAP,pulse,Temp,RR,Fio2,Gender
0,1,34,15.0,93,67.0,75.67,128.0,36.8,14.0,21.0,Female
1,2,20,15.0,130,90.0,103.33,80.0,37.0,16.0,21.0,Male
2,3,77,14.0,163,105.0,124.33,92.0,36.8,18.0,21.0,Female
3,4,23,8.0,100,60.0,73.33,100.0,37.0,12.0,100.0,Female
4,5,86,15.0,150,90.0,110.00,85.0,37.0,19.0,21.0,Female


### Cleaning Method and Error Checks

To clean the `Gender` column, I used a function-based approach instead of directly replacing values in the original column. This allowed me to standardize the values while also checking for common data quality issues.

Several issues were considered during cleaning:

1. **Unmatched values**  
   Any value that did not match the expected gender categories was coded as `-1` for `Unknown/Unspecified`.

2. **Case sensitivity**  
   Since values such as `Male`, `MALE`, and `male` are treated as different strings in Python, I converted entries to lowercase before checking them.

3. **Missing values**  
   Missing or blank entries were handled separately using `pd.isna()` and assigned to `-1`.

4. **Reassignment**  
   The cleaned values were assigned to a new column first, so the original data could be checked before replacing it.

The final coding scheme used was:

- `1 = Male`
- `0 = Female`
- `2 = Non-binary/Other`
- `-1 = Unknown/Unspecified`



### **Alternative Approach Considered**

A dictionary-based mapping could also be used to clean the `Gender` column when all expected values are known in advance. However, I selected the function-based approach as the final method because it is easier to extend and handles missing, unclear or unexpected values more explicitly.

This makes the cleaning process more suitable for clinical datasets, where entries may be inconsistent or incomplete.

### **Save the Cleaned Dataset**

The cleaned dataset is saved to Google Drive so it can be reused for future Week 0 tasks.

In [12]:
OUTPUT_PATH = '/content/drive/MyDrive/Carisurg Portfolio/Week 0/Data/EmergencyTriageDataset_Gender_Cleaned.csv'

df_raw.to_csv(OUTPUT_PATH, index=False)

print(f"Cleaned dataset saved to: {OUTPUT_PATH}")

Cleaned dataset saved to: /content/drive/MyDrive/Carisurg Portfolio/Week 0/Data/EmergencyTriageDataset_Gender_Cleaned.csv


### Day 1 Deliverable Summary

For the Day 1 task, I cleaned the `Gender` column in the Week 0 dataset and documented the process in this notebook.

The completed work includes:

1. Loading the Week 0 dataset into a pandas DataFrame
2. Inspecting the original `Gender` column values
3. Standardizing inconsistent gender entries
4. Handling missing, unclear, or unexpected values
5. Verifying the cleaned results using value counts
6. Saving the cleaned dataset for future Week 0 tasks, ensureing numeric Gender codes were converted back to human-readable labels.

The completed notebook will be pushed to my public `carisurg-portfolio` GitHub repository as part of my Week 0 submission.

---

### Quick Reference

| Concept | Code |
|---------|------|
| Load a CSV | `pd.read_csv('file.csv')` |
| See unique values | `df['col'].unique()` |
| Count each value | `df['col'].value_counts(dropna=False)` |
| Apply cleaning function | `df['col'].apply(function_name)` |
| Drop a column | `df.drop(columns=['col'])` |
| Rename a column | `df.rename(columns={'old': 'new'})` |
| Save cleaned CSV | `df.to_csv('file.csv', index=False)` |
